# 03 · Checkpoint 3 experiments (A100)

Runs **everything** for Checkpoint 3 on one GPU type so the cost numbers are comparable:

| Step | What | Runs |
|---|---|---|
| smoke | `qlora`, `adapters` on 500 English sentences (F1 ≈ 0 is expected) | 2 |
| tune | learning rates for `qlora`, `adapters` on **English dev only** (1e-4, 5e-4, 1e-3) | 6 |
| final `zs` | all 5 methods × seeds 1–3, English only | 15 |
| final `fs1000` | all 5 methods × seeds 1–3, English + 1,000 Swahili sentences | 15 |
| final `fs100` | all 5 methods × seeds 1–3, English + 100 Swahili sentences | 15 |
| table | mean ± std per method and setting | – |

`full_ft`, `head_only` and `lora` keep the learning rates chosen at Checkpoint 2 (in `configs/`).

**How to run:** *Runtime → Change runtime type → A100 GPU*, then run cells 1 and 2. Keep the tab open.
If Colab disconnects, run cells 1 and 2 again: finished runs are skipped.
Results go to `MyDrive/FrozenWeights/results/checkpoint3/`; cell 5 zips them for download.

## 1. Setup (run first in every new session)

In [ ]:
import os, subprocess, sys
from pathlib import Path

BRANCH = "main"
REPO_URL = "https://github.com/Ande404/FrozenWeights.git"
REPO = Path("/content/FrozenWeights")
RESULTS = Path("/content/drive/MyDrive/FrozenWeights/results/checkpoint3")
OUTPUTS = Path("/content/outputs")  # local disk: big training checkpoints are deleted after each run anyway

from google.colab import drive
drive.mount("/content/drive")

def sh(*cmd, check=True):
    subprocess.run(list(cmd), check=check)

if not REPO.exists():
    sh("git", "clone", "-b", BRANCH, REPO_URL, str(REPO))
else:
    sh("git", "-C", str(REPO), "fetch", "origin")
    sh("git", "-C", str(REPO), "checkout", BRANCH)
    sh("git", "-C", str(REPO), "pull", "origin", BRANCH)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"))
# Colab ships an old torchao that makes peft refuse to build LoRA layers. This project does not use it.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "torchao", check=False)

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("commit:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print("GPU:", gpu or "NONE: Runtime > Change runtime type > A100 GPU, then run this cell again")
if gpu and "A100" not in gpu:
    print(f"WARNING: this is a {gpu}, not an A100. Cost numbers are only comparable on one GPU type.")
print("result files so far:", len(list(RESULTS.glob("*.json"))))

## 2. Run everything (about 1–1.5 hours on an A100)

One command does every step in the table above. Progress is printed here and appended to `progress.log` in the results folder.
If some runs fail, the others continue; the failures are listed at the end and their tracebacks are in `failures.log`.

In [ ]:
!cd /content/FrozenWeights && python scripts/run_experiments.py --phase all --results-dir "{RESULTS}" --output-root "{OUTPUTS}"

## 3. Check progress or failures (any time, e.g. after a disconnect)

In [ ]:
for name in ("progress.log", "failures.log"):
    p = RESULTS / name
    print(f"===== {name}")
    print(p.read_text()[-4000:] if p.exists() else "(none yet)")

## 4. Results table

In [ ]:
import pandas as pd
!cd /content/FrozenWeights && python scripts/run_experiments.py --phase table --methods full_ft head_only lora qlora adapters --results-dir "{RESULTS}" > /dev/null
df = pd.read_csv(RESULTS / "summary_all_settings.csv", index_col=[0, 1])
df[["seeds", "lr", "Swahili F1", "Swahili P", "Swahili R", "ORG F1", "English dev F1", "trainable %", "peak GB", "weights GB", "minutes", "checkpoint MB", "GPU"]]

## 5. Zip the results for download

Creates `MyDrive/FrozenWeights/checkpoint3_results.zip` (all result JSONs, predictions, logs and tables, no model weights) and starts a browser download.
Unzip it into `results/checkpoint3/` in the repo.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/drive/MyDrive/FrozenWeights/checkpoint3_results", "zip", RESULTS)
print(zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB,", len(list(RESULTS.iterdir())), "files")
files.download(zip_path)

## Troubleshooting

- **Disconnected:** run cells 1 and 2 again. Finished runs are skipped (each run's JSON is written when it completes).
- **A method failed its smoke test:** the pipeline skips that method and finishes the rest. Send `failures.log` (cell 3); after the fix, re-run cells 1 and 2 and only the missing runs are done.
- **No GPU / wrong GPU:** Runtime → Change runtime type → A100, then cell 1. The runner refuses to train without a GPU.
- **Out of compute units:** stop after a whole setting finishes (`zs`, then `fs1000`, then `fs100`); the table works with whatever is complete.